# CBRE Break – APK Build in Google Colab

Anleitung:
1. Lade diese Datei in Google Colab hoch: https://colab.research.google.com
2. Runtime → Run all
3. APK wird am Ende zum Download angeboten.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq curl git unzip xz-utils zip openjdk-21-jdk python3-pil

In [ ]:
%%bash
set -e
mkdir -p /opt/android-sdk/cmdline-tools /opt/flutter
curl -sL "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip" -o cmdline-tools.zip
unzip -q cmdline-tools.zip -d /opt/android-sdk/cmdline-tools
mv /opt/android-sdk/cmdline-tools/cmdline-tools /opt/android-sdk/cmdline-tools/latest
rm cmdline-tools.zip
yes | /opt/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses || true
/opt/android-sdk/cmdline-tools/latest/bin/sdkmanager "platforms;android-36" "build-tools;36.0.0" "ndk;28.2.13676358" "platform-tools"

In [ ]:
%%bash
set -e
export ANDROID_HOME=/opt/android-sdk
export ANDROID_SDK_ROOT=/opt/android-sdk
export PATH=/opt/flutter/bin:$PATH
curl -sL "https://storage.googleapis.com/flutter_infra_release/releases/stable/linux/flutter_linux_3.44.8-stable.tar.xz" -o flutter.tar.xz
tar -xf flutter.tar.xz -C /opt/flutter --strip-components=1
rm flutter.tar.xz
flutter --version

In [ ]:
%%bash
set -e
pip3 install --no-cache-dir flet==0.25.2 flet-cli==0.25.2
python3 -c "from PIL import Image; print('Pillow OK')"

In [ ]:
%%bash
set -e
git clone https://github.com/yellow-master/cbre-break.git /content/cbre-break
cd /content/cbre-break
git checkout main

In [ ]:
%%bash
set -e
export ANDROID_HOME=/opt/android-sdk
export ANDROID_SDK_ROOT=/opt/android-sdk
export PATH=/opt/flutter/bin:$PATH
cd /content/cbre-break
flet build apk --no-rich-output --project breakapp --product "CBRE Break" --org de.cbre --build-version 1.0 --build-number 1 || true

In [ ]:
%%bash
set -e
cd /content/cbre-break
if [ -d "build/flutter/android" ]; then
  echo "Patching Android project..."
  sed -i 's/version "8.3.1"/version "8.6.0"/' build/flutter/android/settings.gradle
  sed -i 's/gradle-8.7-bin.zip/gradle-8.10.2-bin.zip/' build/flutter/android/gradle/wrapper/gradle-wrapper.properties
  sed -i "s/ext.kotlin_version = '1.9.24'/ext.kotlin_version = '2.0.0'/" build/flutter/android/build.gradle
  sed -i 's/ndkVersion "25.1.8937393"/ndkVersion "28.2.13676358"/' build/flutter/android/app/build.gradle
  python3 -c "
from PIL import Image
import os
src = 'icon.png'
base = 'build/flutter/android/app/src/main/res'
sizes = {'mipmap-mdpi': 48, 'mipmap-hdpi': 72, 'mipmap-xhdpi': 96, 'mipmap-xxhdpi': 144, 'mipmap-xxxhdpi': 192}
img = Image.open(src).convert('RGBA')
for folder, size in sizes.items():
    path = os.path.join(base, folder)
    os.makedirs(path, exist_ok=True)
    resized = img.resize((size, size), Image.LANCZOS)
    resized.save(os.path.join(path, 'ic_launcher.png'))
    print(f'Saved {folder}/ic_launcher.png')
"
  cd build/flutter/android
  chmod +x gradlew
  ./gradlew assembleRelease --no-daemon --console=plain
else
  echo "ERROR: Flutter project not generated"
  exit 1
fi

In [ ]:
from google.colab import files
import os
apk_path = "/content/cbre-break/build/flutter/android/app/release/app-release.apk"
if os.path.exists(apk_path):
    files.download(apk_path)
else:
    print("APK nicht gefunden.")